### Needs `01-Prep-GEPA.ipynb` to be executed first

Reads from:
- `gs_run_ynorm.json`

# Preperations

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [13]:
GEPA_RUN  = "B8_H200mini_V1"
GEPA_PATH = Path(f"GEPA_runs/{GEPA_RUN}")

## Import Data

In [3]:
gs_run_ynorm = pd.read_json(f"{GEPA_RUN}_ynorm.json")
# year_scope_df       = pd.read_json("year_scope_df.json")

CATEGORIES = ["value", "unit"]

In [4]:
CSV_LIST = sorted(
    [p for p in GEPA_PATH.glob("*.csv") if p.stem.isdigit()],
    key=lambda p: int(p.stem)
)

VARIANTS = [i for i in range(len(CSV_LIST))]
# e.g. [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]

In [5]:
eval_log = pd.read_csv(GEPA_PATH / "eval_log.csv")

# The last generation = rows after the final occurrence of run==0
last_gen_start = eval_log[eval_log["run"] == 0].index[-1]
last_gen = eval_log.loc[last_gen_start:].reset_index(drop=True)

BEST_VARIANTS = last_gen[last_gen["is_best"] == True]["run"].tolist()
print(f"Best variants from {GEPA_RUN}: ", BEST_VARIANTS)

Best variants from B8_H200mini_V2:  [0, 2, 6, 13]


#### Cleanup Import

In [6]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)
    
for col in pipeline_cols:
    gs_run_ynorm[col] = gs_run_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [7]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard

def check_hit_exactly(row, extraction_col, gs_col) -> bool:
    # Preparing gold-standard value(s) for comparison
    gs_val  = row[gs_col]
    if isinstance(gs_val,  list):
        gs_set = set(gs_val)
    elif not pd.isna(gs_val):
        gs_set = {gs_val}
    else:
        gs_set = set()
    
    # Preparing extracted value(s) for comparison
    ext_val = row[extraction_col]
    if isinstance(ext_val, list):
        ext_set = set(ext_val)
    elif not pd.isna(ext_val):
        ext_set = {ext_val}
    else:
        ext_set = set()
    
    # Compare value(s), as there could be more than one (e.g., Allianz) or just one (mostly)
    return gs_set == ext_set

In [8]:
def eval_intern(source, result_basis, exact) -> pd.DataFrame:
    result = result_basis.copy()
    if exact:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit_exactly,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    else:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
                
    return result

def eval(source, exact) -> pd.DataFrame:
    
    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]], exact)
    
    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source, exact)
    
    return gs_eval, in_place



def print_eval(gs_eval) -> None: 
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## YNORM

### Exact

In [9]:
ynorm_exact, _ = eval(gs_run_ynorm, True)

print_eval(ynorm_exact)

### value ###
0: True = 1757 || False = 451 || Quota = 79.57%
1: True = 1738 || False = 470 || Quota = 78.71%
2: True = 1778 || False = 430 || Quota = 80.53%
3: True = 1755 || False = 453 || Quota = 79.48%
4: True = 1757 || False = 451 || Quota = 79.57%
5: True = 1773 || False = 435 || Quota = 80.3%
6: True = 1785 || False = 423 || Quota = 80.84%
7: True = 1773 || False = 435 || Quota = 80.3%
8: True = 1715 || False = 493 || Quota = 77.67%
9: True = 1790 || False = 418 || Quota = 81.07%
10: True = 1757 || False = 451 || Quota = 79.57%
11: True = 1755 || False = 453 || Quota = 79.48%
12: True = 1757 || False = 451 || Quota = 79.57%
13: True = 1806 || False = 402 || Quota = 81.79%
14: True = 1806 || False = 402 || Quota = 81.79%
15: True = 1801 || False = 407 || Quota = 81.57%
16: True = 1778 || False = 430 || Quota = 80.53%
17: True = 1785 || False = 423 || Quota = 80.84%
18: True = 1755 || False = 453 || Quota = 79.48%
19: True = 1757 || False = 451 || Quota = 79.57%
20: True = 1778 ||

# Best-Variant Comparison (is_best == True)

In [10]:
def eval_best(source, exact) -> pd.DataFrame:
    """Like eval(), but restricted to BEST_VARIANTS only."""
    result_basis = source[["report_name", "year", "scope"]]
    result = result_basis.copy()
    check_fn = check_hit_exactly if exact else check_hit

    for cat in CATEGORIES:
        for v in BEST_VARIANTS:
            result[f"{v}_{cat}_hit"] = source.apply(
                check_fn, args=(f"{cat}_{v}", cat), axis=1
            )
    return result


def print_eval_best(gs_eval_best) -> None:
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in BEST_VARIANTS:
            col = gs_eval_best[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean() * 100, 2)
            score_row   = last_gen[last_gen["run"] == v].iloc[0]
            print(f"run {v:>2} (GEPA score {score_row['score']:.4f}): "
                  f"True={true_count} | False={false_count} | Quota={quota}%")
        print()

In [11]:

best_ynorm      = eval_best(gs_run_ynorm, exact=False)

print_eval_best(best_ynorm)

### value ###
run  0 (GEPA score 0.7080): True=1772 | False=436 | Quota=80.25%
run  2 (GEPA score 0.7677): True=1778 | False=430 | Quota=80.53%
run  6 (GEPA score 0.8164): True=1801 | False=407 | Quota=81.57%
run 13 (GEPA score 0.8739): True=1806 | False=402 | Quota=81.79%

### unit ###
run  0 (GEPA score 0.7080): True=1780 | False=428 | Quota=80.62%
run  2 (GEPA score 0.7677): True=1786 | False=422 | Quota=80.89%
run  6 (GEPA score 0.8164): True=1801 | False=407 | Quota=81.57%
run 13 (GEPA score 0.8739): True=1806 | False=402 | Quota=81.79%



In [12]:
rows = []
for v in BEST_VARIANTS:
    score_row = last_gen[last_gen["run"] == v].iloc[0]
    for cat in CATEGORIES:
        col = best_ynorm[f"{v}_{cat}_hit"]
        rows.append({
            "run": v,
            "gepa_score": round(score_row["score"], 6),
            "category": cat,
            "eval_type": "ynorm",
            "quota": round(col.mean() * 100, 2),
            "hits": col.value_counts()[True],
        })

best_summary = pd.DataFrame(rows)
best_summary.pivot_table(
    index=["run", "gepa_score"],
    columns=["eval_type", "category"],
    values="quota"
).round(2)

eval_type       ynorm       
category         unit  value
run gepa_score              
0   0.707965    80.62  80.25
2   0.767699    80.89  80.53
6   0.816372    81.57  81.57
13  0.873894    81.79  81.79